# Week 0 — Dataset tour: follow one purchase order

Procure-to-pay in one line: **purchase order** (what we agreed to buy, at what price) → **goods receipt** (what arrived) → **supplier invoice** (what we are asked to pay) → **three-way match** (do all three agree?) → **payment**.

In this tour you follow **PO-1001** through those documents with plain SQL. Harness 1 will do the same thing automatically from Week 3 on.

In [ ]:
# Setup (run first)
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "course-tools @ git+https://github.com/YOUR-GITHUB-ORG/harness-starter@v1#subdirectory=course-tools"],
                   check=True)
    from google.colab import drive
    drive.mount("/content/drive")

import sqlite3
import pandas as pd
import course_tools as ct
from course_tools.helpers import show_pdf

env = ct.bootstrap()
con = sqlite3.connect(env.db)
q = lambda sql, *args: pd.read_sql_query(sql, con, params=args)

## 1. The supplier master

Eight suppliers. Note the **GSTIN** (a 15-character tax ID whose first two digits are the state code) and the **bank details**: an invoice is only paid to the account held here, never to one printed on an invoice.

In [ ]:
q("select supplier_id, name, state, gstin, bank_account, ifsc from suppliers")

## 2. What we ordered: PO-1001

In [ ]:
q("""select l.line_no, l.sku_id, s.description, l.qty, l.unit_price, s.gst_rate
     from po_lines l join skus s using (sku_id) where l.po_id = ?""", "PO-1001")

## 3. What arrived: the goods receipt

In [ ]:
q("""select r.grn_id, r.received_date, g.po_line_no, g.sku_id, g.qty_received
     from goods_receipts r join grn_lines g using (grn_id) where r.po_id = ?""", "PO-1001")

## 4. What we are asked to pay: invoice INV-A

Predict before you look: if everything ordered arrived at the PO price, and GST is 18% within the state, what should the invoice total be?

In [ ]:
show_pdf(env.invoices / "INV-A.pdf")

## 5. Past invoices

Invoices already processed and paid. The duplicate checks in Week 6 compare new invoices with these.

In [ ]:
q("select invoice_id, supplier_id, invoice_number, invoice_date, po_id, total, status from invoices limit 10")

## Your turn

Pick another purchase order from the catalogue (for example the PO on `INV-11`) and repeat steps 2–4. What is different about it? Write one sentence in your notes; you will meet this invoice again in Week 6.

In [ ]:
pd.read_csv(env.data / "catalogue.csv")[["id", "case", "po_reference", "total", "note"]]